In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as lightning
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import DataLoader, random_split, Subset
import warnings
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import os
import torchmetrics

warnings.filterwarnings("ignore")
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.0)

#1. Calculating SHAP Values)
def compute_shap_explanations(model, background_data, test_data, target_class, expected_size=(128, 128)):
    model.eval()
    explainer = shap.GradientExplainer(model, background_data)
    shap_values = explainer.shap_values(test_data, nsamples=50)

    if isinstance(shap_values, list):
        target_shap = shap_values[target_class]
    else:
        target_shap = shap_values

    if isinstance(target_shap, torch.Tensor):
        target_shap = target_shap.cpu().detach().numpy()
        
    heatmap = np.squeeze(target_shap) 

    if heatmap.ndim == 4:
        heatmap = heatmap[..., target_class]

    if heatmap.ndim == 3:
        if heatmap.shape[0] == 3: 
            heatmap = np.sum(heatmap, axis=0)
        elif heatmap.shape[2] == 3: 
            heatmap = np.sum(heatmap, axis=2)

    if heatmap.ndim != 2:
        raise ValueError(f"Shape Error: Unable to reduce SHAP values to 2D map. Current shape: {heatmap.shape}")

    if heatmap.shape != expected_size:
        t_map = torch.tensor(heatmap, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        t_resized = F.interpolate(t_map, size=expected_size, mode='bilinear', align_corners=False)
        heatmap = t_resized.squeeze().detach().numpy()

    return heatmap

# 2. DEMPSTER-SHAFER FUSION 
def phi_to_bpa(phi, model_weight, alpha=100.0):
    phi_norm = np.tanh(phi * alpha)
    m = {'prop': model_weight * max(0.0, phi_norm),
         'anti': model_weight * max(0.0, -phi_norm)}
    m['ign'] = 1.0 - (m['prop'] + m['anti'])
    total = sum(m.values())
    if total > 0: m = {k: v/total for k,v in m.items()}
    else: m = {'prop': 0, 'anti': 0, 'ign': 1.0}
    return m

def dempster_rule_simple(m1, m2):
    K = m1['prop'] * m2['anti'] + m1['anti'] * m2['prop']
    if K >= 0.999: return {'prop': 0, 'anti': 0, 'ign': 1.0}
    norm = 1.0 - K
    m_fused = {}
    m_fused['prop'] = (m1['prop']*m2['prop'] + m1['prop']*m2['ign'] + m1['ign']*m2['prop']) / norm
    m_fused['anti'] = (m1['anti']*m2['anti'] + m1['anti']*m2['ign'] + m1['ign']*m2['anti']) / norm
    m_fused['ign'] = (m1['ign']*m2['ign']) / norm
    return m_fused

def fuse_explanations(explanation_maps, model_weights):
    rows, cols = explanation_maps[0].shape
    bel_map, plaus_map, unc_map = np.zeros((rows, cols)), np.zeros((rows, cols)), np.zeros((rows, cols))

    for i in range(rows):
        for j in range(cols):
            m_fused = {'prop': 0.0, 'anti': 0.0, 'ign': 1.0}
            for idx, exp_map in enumerate(explanation_maps):
                val = exp_map[i, j]
                m_k = phi_to_bpa(float(val), float(model_weights[idx]), alpha=100.0)
                m_fused = dempster_rule_simple(m_fused, m_k)
            
            bel_map[i, j] = m_fused['prop']
            plaus_map[i, j] = 1.0 - m_fused['anti']
            unc_map[i, j] = m_fused['ign'] 
    return bel_map, plaus_map, unc_map

# 3. Bayesian weighing of the models
def evaluate_model_f1(models, dataloader, num_classes, device):
    f1_scores = []
    metric = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="macro").to(device)
    print("\n  Evaluating models based upon F1 Score")
    for idx, model in enumerate(models):
        model.to(device)
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for x, y in dataloader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.append(predicted)
                all_targets.append(y)
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        score = metric(all_preds, all_targets).item()
        f1_scores.append(score)
        print(f"    Model {idx+1} ({model.__class__.__name__}): F1 Score = {score:.4f}")
    return np.array(f1_scores)

def get_bayesian_weights_from_f1(f1_scores, temperature=5.0):
    evidence = f1_scores * 100 
    posterior_alpha = np.ones_like(f1_scores) + (evidence / temperature)
    sampled_weights = np.random.dirichlet(posterior_alpha)
    print(f"\n  [Bayesian Weights Calculated]: {np.round(sampled_weights, 3)}")
    return sampled_weights


class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 16 * 16, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class ViTWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        in_features = self.vit.heads.head.in_features
        self.vit.heads.head = nn.Linear(in_features, num_classes)
        self.resize = transforms.Resize((224, 224), antialias=True)
        
    def forward(self, x):
        x = self.resize(x)
        return self.vit(x)

class UnifiedClassifier(lightning.LightningModule):
    def __init__(self, model_type, num_classes, lr=1e-4): 
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.model_type = model_type
        
        if model_type == "custom":
            self.model = SimpleCNN(num_classes)
        elif model_type == "resnet":
            self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        elif model_type == "vit":
            self.model = ViTWrapper(num_classes)
            
        self.f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="macro")
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        f1_val = self.f1(torch.argmax(logits, 1), y)
        self.log("val_loss", loss)
        self.log("val_f1", f1_val, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}


def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    t = tensor.permute(1, 2, 0).cpu().numpy()
    t = std * t + mean
    return np.clip(t, 0, 1)

def save_visualizations(individual_maps, bel, pl, unc, test_image, model_weights, target_class, class_names, model_names, viz_folder="viz"):
    class_name = class_names[target_class]
    class_folder = os.path.join(viz_folder, class_name)
    os.makedirs(class_folder, exist_ok=True)
    
    orig_img = denormalize(test_image.squeeze())
    all_vals = np.concatenate([m.flatten() for m in individual_maps])
    vmax = np.percentile(np.abs(all_vals), 99) if len(all_vals) > 0 else 1.0

    print(f" Saving visualizations to '{class_folder}/'...")

    fig1 = plt.figure(figsize=(16, 5), constrained_layout=True)
    gs1 = gridspec.GridSpec(1, 4, figure=fig1)
    ax_orig = fig1.add_subplot(gs1[0, 0])
    ax_orig.imshow(orig_img)
    ax_orig.set_title(f"Target: {class_name}", fontweight='bold'); ax_orig.axis('off')
    
    for i in range(3):
        ax = fig1.add_subplot(gs1[0, i+1])
        ax.imshow(orig_img, alpha=1.0)
        heatmap = individual_maps[i]
        masked_map = np.ma.masked_where(np.abs(heatmap) < (vmax * 0.1), heatmap)
        im = ax.imshow(masked_map, cmap='seismic', vmin=-vmax, vmax=vmax, alpha=0.7)
        ax.set_title(f"{model_names[i]} (w={model_weights[i]:.2f})", fontsize=10)
        ax.axis('off')
        if i == 2: plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.savefig(os.path.join(class_folder, f"1_attribution.png"), bbox_inches='tight')
    plt.close(fig1)

    fig2 = plt.figure(figsize=(18, 6), constrained_layout=True)
    gs2 = gridspec.GridSpec(1, 3, figure=fig2)

    # Belief Map
    ax_bel = fig2.add_subplot(gs2[0, 0])
    im_b = ax_bel.imshow(bel, cmap='YlGn', vmin=0, vmax=1)
    ax_bel.set_title("Belief Mass (Support)", fontweight='bold')
    plt.colorbar(im_b, ax=ax_bel, fraction=0.046, pad=0.04); ax_bel.axis('off')

    # Plausibility Map
    ax_pl = fig2.add_subplot(gs2[0, 1])
    im_p = ax_pl.imshow(pl, cmap='Blues', vmin=0, vmax=1)
    ax_pl.set_title("Plausibility (Upper Bound)", fontweight='bold')
    plt.colorbar(im_p, ax=ax_pl, fraction=0.046, pad=0.04); ax_pl.axis('off')

    # Uncertainty Map
    ax_unc = fig2.add_subplot(gs2[0, 2])
    im_u = ax_unc.imshow(unc, cmap='plasma', vmin=0, vmax=1)
    ax_unc.set_title("Uncertainty (Ignorance)", fontweight='bold')
    plt.colorbar(im_u, ax=ax_unc, fraction=0.046, pad=0.04); ax_unc.axis('off')
    
    plt.savefig(os.path.join(class_folder, f"2_fusion.png"), bbox_inches='tight')
    plt.close(fig2)

    fig3 = plt.figure(figsize=(16, 6), constrained_layout=True)
    gs3 = gridspec.GridSpec(1, 2, figure=fig3)

    # 3a. KDE Distribution (Belief vs Uncertainty)
    ax_kde = fig3.add_subplot(gs3[0, 0])
    sns.kdeplot(np.clip(bel.flatten(), 0, 1), fill=True, color='#2ca02c', label='Belief Mass', ax=ax_kde)
    sns.kdeplot(np.clip(pl.flatten(), 0, 1), fill=True, color='#1f77b4', linestyle="--", label='Plausibility', ax=ax_kde)
    sns.kdeplot(np.clip(unc.flatten(), 0, 1), fill=True, color='#6A0DAD', label='Uncertainty', ax=ax_kde)
    
    ax_kde.set_title("Evidence Distribution (Pixel-wise Density)", fontweight='bold')
    ax_kde.set_xlabel("Mass Value (0.0 to 1.0)")
    ax_kde.set_xlim(0, 1)
    ax_kde.legend(loc='upper right')

    ax_bar = fig3.add_subplot(gs3[0, 1])
    y_pos = np.arange(len(model_weights))
    colors = ['#3498db', '#e74c3c', '#9b59b6'] 
    
    bars = ax_bar.barh(y_pos, model_weights, color=colors)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(model_names)
    ax_bar.set_xlabel("Dirichlet Posterior Weight")
    ax_bar.set_title("Bayesian Model Confidence (Based on F1)", fontweight='bold')
    ax_bar.set_xlim(0, 1.0)
    
    for bar in bars:
        width = bar.get_width()
        ax_bar.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center')

    plt.savefig(os.path.join(class_folder, f"3_analytics.png"), bbox_inches='tight')
    plt.close(fig3)

#6. Main function
def main():
    lightning.seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = "./Malaria"
    
    if not os.path.exists(DATA_PATH):
        print(f"ERROR: Folder '{DATA_PATH}' not found.")
        return

    # 1. Data Prep
    print("Loading Dataset...")
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    full_dataset = datasets.ImageFolder(root=DATA_PATH, transform=transform)
    class_names = full_dataset.classes
    num_classes = len(class_names)
    
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_d, test_d = random_split(full_dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_d, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(test_d, batch_size=32, num_workers=0)
    
    model_types = ["custom", "resnet", "vit"]
    trained_models = []
    
    print("\nTraining")
    
    for m_type in model_types:
        print(f"\nTraining Architecture: {m_type.upper()}")
        
        lr = 1e-4 if m_type == "vit" else 1e-3
        model = UnifiedClassifier(model_type=m_type, num_classes=num_classes, lr=lr)
        
        checkpoint_cb = ModelCheckpoint(monitor="val_f1", mode="max", dirpath="checkpoints", filename=f"{m_type}-best")
        early_stop_cb = EarlyStopping(monitor="val_loss", patience=4, mode="min")
        
        trainer = lightning.Trainer(
            max_epochs=1, 
            callbacks=[checkpoint_cb, early_stop_cb],
            accelerator="auto",
            devices=1,
            enable_progress_bar=True,
            log_every_n_steps=5
        )
        
        trainer.fit(model, train_loader, val_loader)
        
        best_model = UnifiedClassifier.load_from_checkpoint(checkpoint_cb.best_model_path)
        trained_models.append(best_model.model) 

    # 3. Bayesian Weighting
    eval_loader = DataLoader(test_d, batch_size=32)
    f1_scores = evaluate_model_f1(trained_models, eval_loader, num_classes, device)
    weights = get_bayesian_weights_from_f1(f1_scores)
    
    # 4. Explanation Loop
    print("\n Generating Explanations ")
    bg_subset = Subset(train_d, range(min(len(train_d), 20))) 
    bg_data = torch.stack([bg_subset[i][0] for i in range(len(bg_subset))]).to(device)

    found_classes = {}
    for i in range(len(test_d)):
        _, label = test_d[i]
        if label not in found_classes: found_classes[label] = i
        if len(found_classes) == num_classes: break
            
    for label, idx in found_classes.items():
        print(f"Processing Class: {class_names[label]}")
        test_img, _ = test_d[idx]
        test_img = test_img.unsqueeze(0).to(device)
        
        maps = []
        for m in trained_models:
            m.to(device)
            maps.append(compute_shap_explanations(m, bg_data, test_img, label))
            
        bel, pl, unc = fuse_explanations(maps, weights)
        save_visualizations(maps, bel, pl, unc, test_img, weights, label, class_names, model_types, viz_folder="Results/Malaria")

    print("\n Experiment completed")

if __name__ == "__main__":
    main()

/Users/DubeyA-Dev/miniconda3/envs/UbiQCon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | model   | SimpleCNN         | 2.1 M  | train
1 | f1      | MulticlassF1Score | 0      | train
2 | loss_fn | CrossEntropyLoss  | 0      | train
------------------------------------------------------
2.1 M     Trainable params
0         Non-trainable params
2.1 M     Total params
8.484     Total estimated model params size (MB)
17        Modules in train mode
0         Modules in eval mode


Loading Dataset...

--- Starting Training Phase ---

Training Architecture: CUSTOM
Epoch 0: 100%|██████████| 689/689 [00:40<00:00, 16.87it/s, v_num=10, train_loss=0.184, val_f1=0.948]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 689/689 [00:40<00:00, 16.85it/s, v_num=10, train_loss=0.184, val_f1=0.948]

Training Architecture: RESNET


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | model   | ResNet            | 11.2 M | train
1 | f1      | MulticlassF1Score | 0      | train
2 | loss_fn | CrossEntropyLoss  | 0      | train
------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.710    Total estimated model params size (MB)
70        Modules in train mode
0         Modules in eval mode


...

Epoch 0: 100%|██████████| 689/689 [01:17<00:00,  8.91it/s, v_num=11, train_loss=0.153, val_f1=0.952]

Training Architecture: VIT


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | model   | ViTWrapper        | 85.8 M | train
1 | f1      | MulticlassF1Score | 0      | train
2 | loss_fn | CrossEntropyLoss  | 0      | train
------------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.201   Total estimated model params size (MB)
156       Modules in train mode
0         Modules in eval mode


Epoch 0: 100%|██████████| 689/689 [17:17<00:00,  0.66it/s, v_num=12, train_loss=0.0625, val_f1=0.945]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 689/689 [17:20<00:00,  0.66it/s, v_num=12, train_loss=0.0625, val_f1=0.945]

  Evaluating Model Robustness (F1 Score)...
    Model 1 (SimpleCNN): F1 Score = 0.9497
    Model 2 (ResNet): F1 Score = 0.9535
    Model 3 (ViTWrapper): F1 Score = 0.9459

  [Bayesian Weights Calculated]: [0.368 0.321 0.311]

--- Generating Explanations ---
Processing Class: Uninfected
    Saving 3 visualizations to 'Results/Malaria/Uninfected/'...
Processing Class: Infected
    Saving 3 visualizations to 'Results/Malaria/Infected/'...

Done. Check 'viz_vit_ensemble/' folder.
